# AgriEstate-Yields: Candidate Land Data Exploration

**Dataset**: `ml/data/candidate_lands.csv` (15 records)

**Purpose**: Explore and understand the candidate land dataset for agricultural suitability assessment.

**Important Limitation**: This dataset contains ONLY candidate land characteristics. It does NOT contain:
- Soil pH, N/P/K measurements
- Rainfall, temperature, humidity
- Crop type or historical yield data
- Actual yield labels

Therefore, this notebook focuses on:
1. Data quality and structure
2. Feature distributions
3. Suitability assessment results (once implemented)
4. Identification of data gaps for future yield modeling

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set up plotting
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Load the dataset
data_path = Path('../data/candidate_lands.csv')
df = pd.read_csv(data_path)
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 1. Dataset Overview

In [ ]:
# Basic info
print("=== SHAPE ===")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

print("\n=== COLUMNS ===")
for col in df.columns:
    print(f"  {col}: {df[col].dtype}")

print("\n=== FIRST 5 ROWS ===")
display(df.head())

In [ ]:
# Check for missing values
print("=== MISSING VALUES ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "No missing values")

# Check for duplicates
print(f"\n=== DUPLICATES ===")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate land_ids: {df['land_id'].duplicated().sum()}")

## 2. Descriptive Statistics

In [ ]:
# Numeric columns statistics
numeric_cols = df.select_dtypes(include=[np.number]).columns
print("=== NUMERIC STATISTICS ===")
display(df[numeric_cols].describe().T)

# Categorical columns
cat_cols = df.select_dtypes(include=['object']).columns
print("\n=== CATEGORICAL COLUMNS ===")
for col in cat_cols:
    if col not in ['land_id', 'name']:
        print(f"\n{col}:")
        print(df[col].value_counts())

## 3. Feature Distributions

In [ ]:
# Area distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Area in sqft
axes[0, 0].hist(df['area_sqft'], bins=10, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Area Distribution (sqft)')
axes[0, 0].set_xlabel('Area (sqft)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].axvline(df['area_sqft'].mean(), color='red', linestyle='--', label=f'Mean: {df["area_sqft"].mean():.0f}')
axes[0, 0].axvline(df['area_sqft'].median(), color='green', linestyle='--', label=f'Median: {df["area_sqft"].median():.0f}')
axes[0, 0].legend()

# Sunlight hours
axes[0, 1].hist(df['sunlight_hours'], bins=10, edgecolor='black', alpha=0.7, color='orange')
axes[0, 1].set_title('Sunlight Hours Distribution')
axes[0, 1].set_xlabel('Sunlight (hours/day)')
axes[0, 1].set_ylabel('Count')
axes[0, 1].axvline(df['sunlight_hours'].mean(), color='red', linestyle='--', label=f'Mean: {df["sunlight_hours"].mean():.1f}')
axes[0, 1].legend()

# Market distance
axes[1, 0].hist(df['market_distance_km'], bins=10, edgecolor='black', alpha=0.7, color='green')
axes[1, 0].set_title('Market Distance Distribution')
axes[1, 0].set_xlabel('Distance (km)')
axes[1, 0].set_ylabel('Count')
axes[1, 0].axvline(df['market_distance_km'].mean(), color='red', linestyle='--', label=f'Mean: {df["market_distance_km"].mean():.1f}')
axes[1, 0].legend()

# Area vs Sunlight scatter
scatter = axes[1, 1].scatter(df['area_sqft'], df['sunlight_hours'], c=df['market_distance_km'], 
                              cmap='viridis', s=100, alpha=0.7, edgecolors='black')
axes[1, 1].set_title('Area vs Sunlight (color = market distance)')
axes[1, 1].set_xlabel('Area (sqft)')
axes[1, 1].set_ylabel('Sunlight (hours)')
plt.colorbar(scatter, ax=axes[1, 1], label='Market Distance (km)')

plt.tight_layout()
plt.show()

In [ ]:
# Categorical distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Land type
land_type_counts = df['land_type'].value_counts()
axes[0].pie(land_type_counts.values, labels=land_type_counts.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Land Type Distribution')

# Current usage
usage_counts = df['current_usage'].value_counts()
axes[1].pie(usage_counts.values, labels=usage_counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Current Usage Distribution')

# Water access
water_counts = df['water_access'].value_counts()
axes[2].pie(water_counts.values, labels=water_counts.index, autopct='%1.1f%%', startangle=90)
axes[2].set_title('Water Access Distribution')

plt.tight_layout()
plt.show()

## 4. Cross-tabulations

In [ ]:
# Land type vs Current usage
print("=== Land Type vs Current Usage ===")
ct1 = pd.crosstab(df['land_type'], df['current_usage'], margins=True)
display(ct1)

# Land type vs Water access
print("\n=== Land Type vs Water Access ===")
ct2 = pd.crosstab(df['land_type'], df['water_access'], margins=True)
display(ct2)

# Current usage vs Water access
print("\n=== Current Usage vs Water Access ===")
ct3 = pd.crosstab(df['current_usage'], df['water_access'], margins=True)
display(ct3)

In [ ]:
# Box plots by category
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Area by land type
sns.boxplot(data=df, x='land_type', y='area_sqft', ax=axes[0, 0])
axes[0, 0].set_title('Area by Land Type')

# Sunlight by land type
sns.boxplot(data=df, x='land_type', y='sunlight_hours', ax=axes[0, 1])
axes[0, 1].set_title('Sunlight by Land Type')

# Market distance by land type
sns.boxplot(data=df, x='land_type', y='market_distance_km', ax=axes[1, 0])
axes[1, 0].set_title('Market Distance by Land Type')

# Area by current usage
sns.boxplot(data=df, x='current_usage', y='area_sqft', ax=axes[1, 1])
axes[1, 1].set_title('Area by Current Usage')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix for numeric features
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, square=True, fmt='.2f')
plt.title('Numeric Feature Correlations')
plt.tight_layout()
plt.show()

## 5. Suitability Assessment Results

Run the suitability engine to see scores for each land parcel.

In [ ]:
# Run suitability assessment
from src.suitability import assess_all_lands, suitability_to_dataframe, get_suitability_summary

results = assess_all_lands()
suit_df = suitability_to_dataframe(results)
summary = get_suitability_summary(results)

print("=== SUITABILITY SUMMARY ===")
for k, v in summary.items():
    print(f"  {k}: {v}")

print("\n=== DETAILED RESULTS ===")
display(suit_df[['land_id', 'name', 'land_type', 'current_usage', 'area_m2', 
                   'suitability_score', 'suitability_status', 'usage_flag', 'limiting_factors']])

In [ ]:
# Visualize suitability scores
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart of suitability scores
colors = {'Highly Suitable': 'green', 'Moderately Suitable': 'yellowgreen', 
          'Marginally Suitable': 'orange', 'Not Suitable': 'red'}
bar_colors = [colors.get(s, 'gray') for s in suit_df['suitability_status']]

axes[0].barh(suit_df['land_id'], suit_df['suitability_score'], color=bar_colors, edgecolor='black')
axes[0].set_xlabel('Suitability Score')
axes[0].set_title('Suitability Scores by Land')
axes[0].invert_yaxis()
axes[0].axvline(0.75, color='green', linestyle='--', alpha=0.5, label='Highly Suitable (0.75)')
axes[0].axvline(0.50, color='yellowgreen', linestyle='--', alpha=0.5, label='Moderately Suitable (0.50)')
axes[0].axvline(0.30, color='orange', linestyle='--', alpha=0.5, label='Marginally Suitable (0.30)')
axes[0].legend()

# Factor scores heatmap
factor_cols = [c for c in suit_df.columns if c.startswith('factor_')]
factor_data = suit_df.set_index('land_id')[factor_cols]
factor_data.columns = [c.replace('factor_', '') for c in factor_data.columns]

sns.heatmap(factor_data.T, annot=True, fmt='.2f', cmap='RdYlGn', center=0.5,
            cbar_kws={'label': 'Factor Score'}, ax=axes[1])
axes[1].set_title('Factor Scores by Land (higher = better)')

plt.tight_layout()
plt.show()

In [ ]:
# Suitability by land type and usage
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=suit_df, x='land_type', y='suitability_score', ax=axes[0])
sns.stripplot(data=suit_df, x='land_type', y='suitability_score', color='black', alpha=0.5, ax=axes[0])
axes[0].set_title('Suitability Score by Land Type')
axes[0].axhline(0.75, color='green', linestyle='--', alpha=0.5)
axes[0].axhline(0.50, color='yellowgreen', linestyle='--', alpha=0.5)
axes[0].axhline(0.30, color='orange', linestyle='--', alpha=0.5)

sns.boxplot(data=suit_df, x='current_usage', y='suitability_score', ax=axes[1])
sns.stripplot(data=suit_df, x='current_usage', y='suitability_score', color='black', alpha=0.5, ax=axes[1])
axes[1].set_title('Suitability Score by Current Usage')
axes[1].axhline(0.75, color='green', linestyle='--', alpha=0.5)
axes[1].axhline(0.50, color='yellowgreen', linestyle='--', alpha=0.5)
axes[1].axhline(0.30, color='orange', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 6. Crop Recommendations (Prototype)

**WARNING**: Recommendations use placeholder crop thresholds that are NOT scientifically validated. Do not use for real decisions.

In [ ]:
# Generate crop recommendations for all available lands
from src.recommendation import recommend_for_all_lands, recommendations_to_dataframe

all_recs = recommend_for_all_lands(top_n=3)
rec_df = recommendations_to_dataframe(all_recs)

print("=== CROP RECOMMENDATIONS (TOP 3 PER LAND) ===")
display(rec_df[['land_id', 'land_name', 'crop', 'crop_match_score', 
                   'estimated_yield_kg', 'estimated_revenue_usd', 'data_gaps']])

In [ ]:
# Visualize crop recommendations
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top crop match scores
top_crops = rec_df.drop_duplicates(subset='land_id', keep='first')
axes[0].barh(top_crops['land_id'], top_crops['crop_match_score'], 
             color='teal', edgecolor='black')
axes[0].set_xlabel('Crop Match Score')
axes[0].set_title('Best Crop Match Score per Land')
axes[0].invert_yaxis()

# Estimated revenue by crop
sns.barplot(data=rec_df, x='estimated_revenue_usd', y='crop', hue='land_id', 
            ax=axes[1], dodge=False, estimator='sum', errorbar=None)
axes[1].set_title('Total Estimated Revenue by Crop (across all lands)')
axes[1].set_xlabel('Estimated Revenue (USD)')
axes[1].legend(title='Land', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

## 7. Data Gaps Analysis

Identify what data is missing for yield prediction and improved recommendations.

In [ ]:
# Required features for yield model (from model.py)
from src.model import EXPECTED_NUMERIC_FEATURES, EXPECTED_CATEGORICAL_FEATURES, TARGET_COLUMN

print("=== FEATURES AVAILABLE IN CURRENT DATASET ===")
available = [c for c in df.columns if c not in ['land_id', 'name']]
for c in available:
    print(f"  \u2713 {c}")

print("\n=== FEATURES REQUIRED FOR YIELD MODEL (NOT IN CURRENT DATA) ===")
for f in EXPECTED_NUMERIC_FEATURES:
    status = "\u2713" if f in df.columns else "\u2717 MISSING"
    print(f"  {status} {f}")

for f in EXPECTED_CATEGORICAL_FEATURES:
    status = "\u2713" if f in df.columns else "\u2717 MISSING"
    print(f"  {status} {f}")

print(f"\n=== TARGET VARIABLE ===")
print(f"  \u2717 MISSING: {TARGET_COLUMN}")

In [ ]:
# Summary of data gaps for each land
print("=== DATA GAPS FOR YIELD PREDICTION ===")
print("Each land needs the following measured/collected:")
gaps = [
    "Soil pH (lab test)",
    "Soil N (nitrogen) - kg/ha or ppm",
    "Soil P (phosphorus) - kg/ha or ppm",
    "Soil K (potassium) - kg/ha or ppm",
    "Historical rainfall (growing season, mm)",
    "Average temperature (growing season, \u00b0C)",
    "Average humidity (growing season, %)",
    "Water availability (irrigation capacity, mm)",
    "Crop type to be grown",
    "Soil type classification",
    "Irrigation type",
    "Growing season",
    "ACTUAL YIELD (kg/ha or kg/m2) - for training labels"
]
for i, gap in enumerate(gaps, 1):
    print(f"  {i:2d}. {gap}")

## 8. Export Processed Data

In [ ]:
# Export preprocessed data with suitability scores
from src.preprocessing import preprocess_candidate_lands

processed_df = preprocess_candidate_lands()
processed_df = processed_df.merge(suit_df[['land_id', 'suitability_score', 'suitability_status', 
                                            'usage_flag', 'limiting_factors']], on='land_id')

output_path = Path('../data/candidate_lands_processed.csv')
processed_df.to_csv(output_path, index=False)
print(f"Processed data saved to: {output_path}")
print(f"Shape: {processed_df.shape}")
display(processed_df.head())

In [ ]:
# Calculate summary statistics dynamically
usage_counts = df['current_usage'].value_counts()
land_type_counts = df['land_type'].value_counts()
water_counts = df['water_access'].value_counts()

print("=== KEY FINDINGS (DYNAMICALLY CALCULATED) ===")
print(f"- {len(df)} candidate lands: {land_type_counts.get('Rooftop', 0)} Rooftops, {land_type_counts.get('Plot', 0)} Plots")
usage_str = ", ".join([f"{usage_counts.get(k, 0)} {k}" for k in ['Vacant', 'Underutilized', 'Existing Garden'] if usage_counts.get(k, 0) > 0])
print(f"- Current usage: {usage_str}")
print(f"- Sunlight range: {df['sunlight_hours'].min()} - {df['sunlight_hours'].max()} hours (all reasonable for agriculture)")
water_str = ", ".join([f"{water_counts.get(k, 0)} {k}" for k in ['High', 'Medium', 'Low'] if water_counts.get(k, 0) > 0])
print(f"- Water access: {water_str}")
print(f"- Market distance: {df['market_distance_km'].min()} - {df['market_distance_km'].max()} km")
print(f"- Area range: {df['area_sqft'].min():,} - {df['area_sqft'].max():,} sqft ({df['area_sqft'].min() * 0.092903:.0f} - {df['area_sqft'].max() * 0.092903:.0f} m2)")

print("\n=== SUITABILITY RESULTS ===")
print("- Run the suitability engine to see rankings")
print("- Plots generally score higher due to land_type_base (1.0 vs 0.85)")
print("- Vacant lands score higher on usage_priority")

print("\n=== NEXT STEPS FOR YIELD PREDICTION ===")
print("1. Collect soil samples (pH, N, P, K) for each land")
print("2. Obtain historical weather data (rainfall, temperature, humidity)")
print("3. Define crop types for each land")
print("4. Conduct trial cultivation to measure ACTUAL YIELD")
print("5. Build labeled dataset with 50+ samples minimum")
print("6. Train RandomForestRegressor using `model.py`")

print("\n=== CROP RECOMMENDATIONS ===")
print("- Current recommendations are PROTOTYPE only")
print("- Require validated crop requirement thresholds")
print("- Need soil pH, temperature, rainfall data per land")